# limpieza de datos

In [1]:
import pandas as pd

from config import CRUDO, PROCESADO, ANIOS

In [2]:
def leer_anio(anio):
    """Lee la base anual de importaciones de un año y la verifica contra el archivo de control.

    Supone que el zip ya se descomprimió en datos/crudo/imports_{anio}/
    y que los archivos se llaman impoaAA.csv e itotaAA.csv (AA = últimos dos dígitos del año).
    """
    carpeta = CRUDO / f"imports_{anio}"
    aa = str(anio)[-2:]

    datos = pd.read_csv(
        carpeta / f"impoa{aa}.csv",
        sep=";",
        decimal=",",
        encoding="latin1",
        dtype={"NCM": str, "Porg": str},
    )
    control = pd.read_csv(
        carpeta / f"itota{aa}.csv",
        sep=";",
        decimal=",",
        encoding="latin1",
    )

    # Entre 2019 y 2021 el INDEC usa "(u$s)" y desde 2022 "(USD)": unifico al formato nuevo
    datos.columns = datos.columns.str.replace("(u$s)", "(USD)", regex=False)
    control.columns = control.columns.str.replace("(u$s)", "(USD)", regex=False)

    # Verificación contra los totales de control del INDEC
    assert len(datos) == control.loc[0, "Total de registros"], f"{anio}: no coincide la cantidad de filas"
    assert abs(datos["CIF(USD)"].sum() - control.loc[0, "Total CIF(USD)"]) < 1, f"{anio}: no coincide el total CIF"

    return datos

for anio in ANIOS:
    aa = str(anio)[-2:]
    carpeta = CRUDO / f"imports_{anio}"
    datos_cols = pd.read_csv(carpeta / f"impoa{aa}.csv", sep=";", encoding="latin1", nrows=0).columns
    control_cols = pd.read_csv(carpeta / f"itota{aa}.csv", sep=";", encoding="latin1", nrows=0).columns
    print(anio)
    print("  datos:  ", list(datos_cols))
    print("  control:", list(control_cols))

2019
  datos:   ['Año', 'NCM', 'Porg', 'Pnet(kg)', 'FOB(u$s)', 'Flete(u$s)', 'Seguro(u$s)', 'CIF(u$s)']
  control: ['Tabla', 'Año', 'periodicidad', 'Total de registros', 'Total Pnet(kg)', 'Total FOB(u$s)', 'Total Flete(u$s)', 'Total Seguro(u$s)', 'Total CIF(u$s)', 'Unnamed: 9']
2020
  datos:   ['Año', 'NCM', 'Porg', 'Pnet(kg)', 'FOB(u$s)', 'Flete(u$s)', 'Seguro(u$s)', 'CIF(u$s)']
  control: ['Tabla', 'Año', 'periodicidad', 'Total de registros', 'Total Pnet(kg)', 'Total FOB(u$s)', 'Total Flete(u$s)', 'Total Seguro(u$s)', 'Total CIF(u$s)', 'Unnamed: 9']
2021
  datos:   ['Año', 'NCM', 'Porg', 'Pnet(kg)', 'FOB(u$s)', 'Flete(u$s)', 'Seguro(u$s)', 'CIF(u$s)']
  control: ['Tabla', 'Año', 'periodicidad', 'Total de registros', 'Total Pnet(kg)', 'Total FOB(u$s)', 'Total Flete(u$s)', 'Total Seguro(u$s)', 'Total CIF(u$s)', 'Unnamed: 9']
2022
  datos:   ['Año', 'NCM', 'Porg', 'Pnet(kg)', 'FOB(USD)', 'Flete(USD)', 'Seguro(USD)', 'CIF(USD)']
  control: ['Tabla', 'Año', 'periodicidad', 'Total de regis

In [3]:
bases = []
for anio in ANIOS:
    bases.append(leer_anio(anio))
    print(anio, "ok")

impo = pd.concat(bases, ignore_index=True)
print(impo.shape)
impo.head()

2019 ok
2020 ok
2021 ok
2022 ok
2023 ok
2024 ok
2025 ok
(490822, 8)


,Año,NCM,Porg,Pnet(kg),FOB(USD),Flete(USD),Seguro(USD),CIF(USD)
0,2019,01012100,200,9800.0,354774.00,45675.01,3636.86,404085.87
1,2019,01012100,203,11350.0,174203.39,19332.02,1683.94,195219.35
2,2019,01012100,208,11000.0,136291.00,6294.63,2725.82,145311.45
3,2019,01012100,212,8300.0,449989.50,46382.80,3226.60,499598.90
4,2019,01012100,218,500.0,35000.00,1724.64,367.25,37091.89


In [4]:
print(impo.dtypes)

assert set(impo["Año"]) == set(ANIOS), "Faltan o sobran años"
assert not impo.duplicated(["Año", "NCM", "Porg"]).any(), "Hay filas repetidas"
assert impo["CIF(USD)"].ge(0).all(), "Hay montos negativos"

Año              int64
NCM                str
Porg               str
Pnet(kg)       float64
FOB(USD)       float64
Flete(USD)     float64
Seguro(USD)    float64
CIF(USD)       float64
dtype: object


In [5]:
paises = pd.read_csv(
    CRUDO / "paises.csv",
    sep=";",
    dtype=str,
    keep_default_na=False,  # para que el código "NA" de Namibia no se lea como dato faltante
)
paises = paises.rename(columns={"_id": "Porg"})  # mismo nombre que en la base de importaciones

print(paises.shape)
paises.head()

(280, 3)


,Porg,iso2,nombre
0,301,AF,Afganistán
1,401,AL,Albania
2,404,AD,Andorra
3,149,AO,Angola
4,237,AG,Antigua y Barbuda


In [6]:
assert paises["Porg"].is_unique, "Hay códigos de país repetidos en el nomenclador"
assert (paises["iso2"] != "").all(), "Hay países sin código ISO"
assert set(impo["Porg"]).issubset(set(paises["Porg"])), "Hay códigos en la base que no están en el nomenclador"

In [7]:
import re

def es_codigo_especial(iso2):
    """Indica si un código ISO de dos letras NO corresponde a un país.

    La norma ISO 3166 reserva AA, QM a QZ, XA a XZ y ZZ para usos propios.
    El INDEC los usa para entradas como 'Confidencial', 'Indeterminado',
    'Resto' o 'Sin declarar'. Supone que el código viene en mayúsculas.
    """
    return re.search(r"^(AA|Q[M-Z]|X[A-Z]|ZZ)$", iso2) is not None

In [8]:
casos = {
    "ZZ": True,   # Confidencial
    "XR": True,   # Indeterminado (continente)
    "QZ": True,   # Indeterminado (América)
    "BR": False,  # Brasil
    "CN": False,  # China
    "QA": False,  # Qatar: empieza con Q pero no está en el rango reservado
    "NA": False,  # Namibia
    "": False,
}
for texto, esperado in casos.items():
    obtenido = es_codigo_especial(texto)
    assert obtenido == esperado, f"{texto}: se esperaba {esperado} y se obtuvo {obtenido}"
print("Los", len(casos), "casos pasan")

Los 8 casos pasan


In [9]:
paises["especial"] = paises["iso2"].map(es_codigo_especial)
print(paises["especial"].value_counts())
paises.loc[paises["especial"], ["Porg", "iso2", "nombre"]]

especial
False    247
True      33
Name: count, dtype: int64


,Porg,iso2,nombre
33,994,XO,Calamares
46,999,ZZ,Confidencial
74,996,XP,Exportaciones e Importaciones Simplificadas
98,398,XC,Indeterminado ( Asia )
99,298,QZ,Indeterminado (América)
100,498,XF,Indeterminado (Europa)
101,598,XM,Indeterminado (Oceanía)
102,998,XR,Indeterminado (continente)
103,198,QQ,Indeterminado (África)
145,295,QW,Mar Territorial Argentino y/o Zona Económica E...


In [10]:
filas_antes = len(impo)

impo = impo.merge(paises, on="Porg", how="left", validate="m:1", indicator=True)

print("Filas antes:", filas_antes, "| Filas después:", len(impo))
print(impo["_merge"].value_counts())

assert len(impo) == filas_antes, "El merge cambió la cantidad de filas"
assert (impo["_merge"] == "both").all(), "Hay códigos de país sin pareja en el nomenclador"

impo = impo.drop(columns="_merge")

Filas antes: 490822 | Filas después: 490822
_merge
both          490822
left_only          0
right_only         0
Name: count, dtype: int64


In [11]:
print(impo["especial"].value_counts())

cif_total = impo.groupby("Año")["CIF(USD)"].sum()
cif_especial = impo[impo["especial"]].groupby("Año")["CIF(USD)"].sum()

print((cif_especial / cif_total * 100).round(2))

especial
False    490663
True        159
Name: count, dtype: int64
Año
2019    1.19
2020    1.07
2021    1.05
2022    1.09
2023    1.07
2024    0.94
2025    1.59
Name: CIF(USD), dtype: float64


Las entradas que no son países (“Confidencial”, “Indeterminado”, etc.) son 159 filas y representan entre 0,9% y 1,6% del valor importado cada año. No se pueden ubicar en el mapa, así que las dejo afuera del análisis por país, pero informo su peso. Al ser menos del 2% en todos los años, no deberían cambiar las conclusiones.

In [12]:
def es_ncm_valido(ncm):
    """Indica si un código NCM tiene el formato correcto: exactamente 8 dígitos.

    Sirve para detectar códigos que perdieron el cero inicial (por haberse leído
    como número) o que tienen letras o espacios. Supone que el código es texto.
    """
    return re.search(r"^\d{8}$", ncm) is not None

In [13]:
casos_ncm = {
    "01012100": True,    # caballos reproductores, con su cero inicial
    "87032310": True,    # autos
    "1012100": False,    # el mismo código si perdió el cero: 7 dígitos
    "0101210A": False,   # tiene una letra
    "01012100 ": False,  # tiene un espacio al final
    "": False,
}
for texto, esperado in casos_ncm.items():
    obtenido = es_ncm_valido(texto)
    assert obtenido == esperado, f"{texto}: se esperaba {esperado} y se obtuvo {obtenido}"
print("Los", len(casos_ncm), "casos pasan")

Los 6 casos pasan


In [14]:
validos = impo["NCM"].map(es_ncm_valido)
print("Códigos válidos:", validos.sum(), "| No válidos:", (~validos).sum())
assert validos.all(), "Hay códigos NCM con formato incorrecto"

Códigos válidos: 490822 | No válidos: 0


In [15]:
# Saco las entradas que no son países (Confidencial, Indeterminado, etc.)
impo_paises = impo[~impo["especial"]]

# Sumo el valor importado por año y país
por_pais = impo_paises.groupby(["Año", "iso2"], as_index=False)["CIF(USD)"].sum()

print(por_pais.shape)
por_pais.head()

(1272, 3)


,Año,iso2,CIF(USD)
0,2019,AD,2.820360e+03
1,2019,AE,1.860961e+08
2,2019,AF,2.665620e+04
3,2019,AG,7.798900e+02
4,2019,AL,7.226068e+05


In [16]:
total_anio = por_pais.groupby("Año")["CIF(USD)"].sum()
por_pais["participacion"] = por_pais["CIF(USD)"] / por_pais["Año"].map(total_anio) * 100

por_pais.sort_values(["Año", "participacion"], ascending=[True, False]).head(10)

,Año,iso2,CIF(USD),participacion
22,2019,BR,1.015933e+10,20.928787
36,2019,CN,9.258744e+09,19.073529
176,2019,US,6.217043e+09,12.807455
44,2019,DE,2.765873e+09,5.697852
143,2019,PY,1.646757e+09,3.392412
21,2019,BO,1.368970e+09,2.820155
165,2019,TH,1.184467e+09,2.440068
83,2019,IT,1.125974e+09,2.319569
122,2019,MX,1.125233e+09,2.318042
53,2019,ES,9.772216e+08,2.013131


In [17]:
assert not por_pais.duplicated(["Año", "iso2"]).any(), "Hay un país repetido en el mismo año"
assert set(por_pais["Año"]) == set(ANIOS), "Faltan años"

suma = por_pais.groupby("Año")["participacion"].sum()
assert ((suma - 100).abs() < 0.001).all(), "Las participaciones no suman 100% en algún año"

PROCESADO.mkdir(parents=True, exist_ok=True)
por_pais.to_csv(PROCESADO / "importaciones_por_pais.csv", index=False)